# CardioSense: Model Training on UCI Cleveland Benchmark Cohort
**Dataset**: Canonical UCI Cleveland Heart Disease Cohort ($N=303$, Janosi et al., 1988)
**Architectures**: Logistic Regression ($L_2$), Random Forest (Bagging), XGBoost (Gradient Boosting)
**Purpose**: Preserved historical benchmark to evaluate model performance across Western clinical predictors (including `thal`).


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import joblib
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

from src.config import get_dataset_config, PARAM_GRIDS, RANDOM_STATE, CV_FOLDS, MODELS_DIR
from src.data_loader import split_and_save_data
from src.preprocessing import build_pipeline


## 1. Load and Partition UCI Cleveland Benchmark


In [ ]:
train_df, test_df = split_and_save_data(dataset="uci")
print(f"UCI Training partition: {train_df.shape} | Test partition: {test_df.shape}")

cfg = get_dataset_config("uci")
X_train = train_df[cfg["feature_names"]]
y_train = train_df[cfg["target_column"]]
X_test = test_df[cfg["feature_names"]]
y_test = test_df[cfg["target_column"]]


## 2. Cross-Validation and Model Training


In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# 1. Logistic Regression
pipe_lr = build_pipeline(LogisticRegression(random_state=RANDOM_STATE), dataset="uci", include_scaling=True)
grid_lr = GridSearchCV(pipe_lr, PARAM_GRIDS["logistic_regression"], cv=cv, scoring="roc_auc", n_jobs=-1)
grid_lr.fit(X_train, y_train)

# 2. Random Forest
pipe_rf = build_pipeline(RandomForestClassifier(random_state=RANDOM_STATE), dataset="uci", include_scaling=False)
grid_rf = GridSearchCV(pipe_rf, PARAM_GRIDS["random_forest"], cv=cv, scoring="roc_auc", n_jobs=-1)
grid_rf.fit(X_train, y_train)

# 3. XGBoost
pipe_xgb = build_pipeline(XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"), dataset="uci", include_scaling=False)
grid_xgb = GridSearchCV(pipe_xgb, PARAM_GRIDS["xgboost"], cv=cv, scoring="roc_auc", n_jobs=-1)
grid_xgb.fit(X_train, y_train)

print(f"Logistic Regression CV ROC-AUC: {grid_lr.best_score_:.4f}")
print(f"Random Forest Best CV ROC-AUC: {grid_rf.best_score_:.4f} (Champion)")
print(f"XGBoost Best CV ROC-AUC: {grid_xgb.best_score_:.4f}")


## 3. Benchmark Evaluation on Held-Out Test Set (N=61)


In [ ]:
models = {
    "Logistic Regression": grid_lr.best_estimator_,
    "Random Forest": grid_rf.best_estimator_,
    "XGBoost": grid_xgb.best_estimator_
}

results = []
for name, model in models.items():
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, preds)
    tn, fp, fn, tp = cm.ravel()
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "F1-Score": f1_score(y_test, preds),
        "ROC-AUC": roc_auc_score(y_test, probs),
        "FN": fn,
        "FP": fp
    })

pd.DataFrame(results)


## 4. Persist UCI Benchmark Champion Artifact


In [ ]:
save_dir = MODELS_DIR / "uci"
save_dir.mkdir(parents=True, exist_ok=True)

champion_uci = grid_rf.best_estimator_
artifact_path = save_dir / "best_pipeline.joblib"
joblib.dump(champion_uci, artifact_path)
print(f"Saved UCI champion pipeline to: {artifact_path}")
